# Phần 3 — VLM & Prompting: chạy thử trên Kaggle

Notebook chạy `generate_json()` trên keyframe, sinh JSON metadata mô tả ảnh.

**Trước khi Run All, bật 2 thứ ở panel Settings bên phải:**
1. **Accelerator** → `GPU T4 x2` hoặc `GPU P100`
2. **Internet** → `On` (cần để tải model từ HuggingFace)

Thiếu GPU thì model chạy trên CPU — chậm gấp hàng chục lần.


## 1. Kiểm tra GPU

Luôn kiểm tra trước. Nếu ô này báo không có GPU, dừng lại và bật Accelerator.


In [ ]:
import torch

print('PyTorch :', torch.__version__)
print('Co GPU  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Ten GPU :', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'VRAM    : {vram:.1f} GB')
else:
    print('CANH BAO: chua bat GPU. Settings > Accelerator > GPU')


## 2. Cài thư viện

Kaggle có sẵn `torch` và `transformers`. Chỉ cần cài thêm phần lượng tử hóa 4-bit.
Mất khoảng 1-2 phút.


In [ ]:
!pip install -q -U transformers accelerate bitsandbytes qwen-vl-utils pydantic
print('Cai xong')


## 3. Nạp code Phần 3

Hai cách, ô dưới tự thử lần lượt:
- **Cách A**: clone repo nhóm (cần repo public hoặc đã cấu hình token)
- **Cách B**: upload thư mục `vlm_prompting` làm Kaggle Dataset rồi Add Data


In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/lolizabrett-byte/Multimodal-Agentic-Retrieval-Engine.git'
BRANCH   = 'research/vlm-prompting'
DICH     = Path('/kaggle/working/repo')

if not DICH.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, str(DICH)],
                   check=False)

PKG = DICH / 'system1' / 'research' / 'vlm_prompting'

if not PKG.exists():
    ung_vien = list(Path('/kaggle/input').glob('**/vlm_prompting'))
    if ung_vien:
        PKG = ung_vien[0]

assert PKG.exists(), f'Khong tim thay code tai {PKG}. Dung cach B: upload dataset.'
sys.path.insert(0, str(PKG))
print('Code tai:', PKG)


In [ ]:
from vlm import generate_json, thong_tin_moi_truong, goi_y_theo_vram, MODEL_REGISTRY

moi_truong = thong_tin_moi_truong()
print(moi_truong)

if moi_truong['vram_gb']:
    goi_y = goi_y_theo_vram(moi_truong['vram_gb'])
    print()
    print('Model chay duoc tren GPU nay:')
    for k in goi_y:
        s = MODEL_REGISTRY[k]
        print(f'  {k:<14} {s.vram_4bit_gb:>4.1f}GB  {s.ten_hien_thi}')


## 4. Chuẩn bị ảnh test

Ba nguồn, thử lần lượt:
1. Ảnh rời trong `/kaggle/input` (dataset đã Add Data)
2. File `.blob` — đọc thẳng 1 ảnh trong kho nén, **không giải nén cả kho** (tiết kiệm đĩa)
3. Ảnh mẫu tải từ internet — luôn chạy được


In [ ]:
import glob, zipfile
import io as _io
from PIL import Image

anh_test = None
nguon = None
DUOI_ANH = ('.jpg', '.jpeg', '.png', '.webp')

for duoi in ('jpg', 'jpeg', 'png', 'webp'):
    tim = glob.glob(f'/kaggle/input/**/*.{duoi}', recursive=True)
    if tim:
        anh_test = Image.open(tim[0]).convert('RGB')
        nguon = f'anh roi: {tim[0]}'
        break

if anh_test is None:
    blobs = glob.glob('/kaggle/input/**/*.blob', recursive=True)
    if blobs:
        zf = zipfile.ZipFile(blobs[0])
        ten = [n for n in zf.namelist() if n.lower().endswith(DUOI_ANH)]
        if ten:
            anh_test = Image.open(_io.BytesIO(zf.read(ten[0]))).convert('RGB')
            nguon = f'blob: {blobs[0]} -> {ten[0]}'

if anh_test is None:
    import urllib.request
    url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
    urllib.request.urlretrieve(url, '/kaggle/working/anh_mau.jpg')
    anh_test = Image.open('/kaggle/working/anh_mau.jpg').convert('RGB')
    nguon = 'anh mau COCO tai tu internet'

print('Nguon anh:', nguon)
print('Kich thuoc:', anh_test.size)
anh_test


## 5. Chạy thử MỘT ảnh

Bước quan trọng nhất. Một ảnh chạy vài giây; prompt sai thì sửa rồi chạy lại ngay.
**Đừng chạy 100 ảnh trước khi ô này ra kết quả sạch.**

Lần đầu sẽ tải model về (vài phút, model nặng 1-15GB tùy loại).


In [ ]:
import json, time
from vlm.generate import reset_vram_counter

MODEL = 'qwen25vl-3b'   # doi thanh 'vintern-1b' de thu model chuyen tieng Viet

reset_vram_counter()
bat_dau = time.time()

ket_qua = generate_json(anh_test, model_key=MODEL, debug=True)

print('=' * 60)
print(json.dumps(ket_qua, ensure_ascii=False, indent=2))
print('=' * 60)
print('Thoi gian:', ket_qua['_latency_sec'], 's')
print('VRAM dinh:', ket_qua.get('_vram_peak_gb'), 'GB')
print(f'Tong ke ca nap model: {time.time() - bat_dau:.1f}s')


## 6. Chế độ DEBUG — soi vài ảnh

Chạy 3-5 ảnh, in đầy đủ để kiểm tra bằng mắt trước khi chạy hàng loạt.

*Không phải cấu hình nào cũng phù hợp — phải dò trước khi đốt giờ GPU.*


In [ ]:
SO_ANH_DEBUG = 3

danh_sach = []
for duoi in ('jpg', 'jpeg', 'png', 'webp'):
    danh_sach += glob.glob(f'/kaggle/input/**/*.{duoi}', recursive=True)
danh_sach = danh_sach[:SO_ANH_DEBUG]

if not danh_sach:
    danh_sach = ['/kaggle/working/anh_mau.jpg']

for i, duong_dan in enumerate(danh_sach, 1):
    print(f'--- Anh {i}/{len(danh_sach)}: {duong_dan} ---')
    try:
        r = generate_json(duong_dan, model_key=MODEL)
        print('  doi tuong :', r['doi_tuong'])
        print('  mau sac   :', r['mau_sac'])
        print('  hanh dong :', r['hanh_dong'])
        print('  boi canh  :', r['boi_canh'])
        print('  caption   :', r['caption_chi_tiet'])
        print('  latency   :', r['_latency_sec'], 's')
    except Exception as e:
        print(f'  LOI: {type(e).__name__}: {e}')
    print()


## 7. So sánh các model (yêu cầu của đề bài)

Đề bài yêu cầu benchmark ít nhất 3 model. Ô này chạy cùng một ảnh qua nhiều model.

⚠️ Mỗi model tải 1-15GB. Chạy hết sẽ tốn thời gian và dung lượng đĩa Kaggle.


In [ ]:
DANH_SACH_MODEL = ['vintern-1b', 'qwen2vl-2b', 'qwen25vl-3b']

bang_ket_qua = []
for ten_model in DANH_SACH_MODEL:
    print(f'=== {ten_model} ===')
    reset_vram_counter()
    try:
        r = generate_json(anh_test, model_key=ten_model)
        bang_ket_qua.append({
            'model': ten_model,
            'latency_s': r['_latency_sec'],
            'vram_gb': r.get('_vram_peak_gb'),
            'json_hop_le': True,
            'caption': r['caption_chi_tiet'][:100],
        })
        print('  OK ', r['_latency_sec'], 's |', r['caption_chi_tiet'][:80])
    except Exception as e:
        bang_ket_qua.append({
            'model': ten_model,
            'json_hop_le': False,
            'loi': f'{type(e).__name__}: {str(e)[:100]}',
        })
        print(f'  LOI: {type(e).__name__}: {str(e)[:100]}')
    print()

import pandas as pd
pd.DataFrame(bang_ket_qua)


## 8. Lưu kết quả

File trong `/kaggle/working/` tải về được ở tab Output sau khi notebook chạy xong.

⚠️ **Phiên Kaggle tự ngắt sau 12 giờ.** Chạy hàng loạt phải lưu mỗi 25 ảnh,
không thì mất trắng cả phiên.


In [ ]:
from pathlib import Path

OUT = Path('/kaggle/working/vlm_smoke_results.json')
OUT.write_text(json.dumps({
    'moi_truong': moi_truong,
    'model_mac_dinh': MODEL,
    'ket_qua_mot_anh': ket_qua,
    'so_sanh_model': bang_ket_qua,
}, ensure_ascii=False, indent=2), encoding='utf-8')

print('Da luu:', OUT)
print('Tai ve o tab Output ben phai sau khi notebook chay xong.')
